<a href="https://colab.research.google.com/github/yabunayyah/Statistic/blob/main/Ordinary%20Linear%20Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Environmental and Library Preparation


Install some libraries

In [ ]:
!pip install torch==2.9.0  numpy==2.0.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 791.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32

Import library

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import TensorDataset, DataLoader, random_split


Preparing Synthetic Data

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

num_samples = 1000

# Feature X (eg: work experience in (years) and max 10 years), range 0-10
X_numpy = np.random.rand(num_samples, 1) * 10

# Target y (eg: salary in millions of rupiah(Rp) ), formula : y = 5x + 1 + noise
noise = np.random.rand(num_samples, 1) * 3
y_numpy = 5 * X_numpy + 1 + noise

# Converting to PyTorch Tensor
X = torch.tensor(X_numpy, dtype=torch.float32)
y = torch.tensor(y_numpy, dtype=torch.float32)

# Dataset Splitting (Train/Val Split)
dataset = TensorDataset(X, y)
train_size = int(0.8 * num_samples)
val_size = num_samples - train_size

# random split
train_dataset, val_dataset = random_split(dataset,[train_size, val_size])

# DataLoader Initialization
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

print(f"Total Data Training: {len(train_dataset)} sample")
print(f"Total Data Validasi: {len(val_dataset)} sample")


### 3. PyTorch Version

Making a simple model

In [ ]:
class OrdinaryLinearRegressionTorch(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(1,1)

    def forward(self, x):
        return self.layer(x)

# Initialize Model
model = OrdinaryLinearRegressionTorch()

Training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
epochs = 100

print("Training ...")

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

print("Training finish!")

Visualization

In [ ]:
import matplotlib.pyplot as plt

model.eval()

# Get one batch of validation data
x_val_batch, y_val_batch = next(iter(val_loader))

# Move to the same device as the model for prediction
x_val_device = x_val_batch.to(device)

with torch.no_grad():
    y_pred = model(x_val_device)

# Move back to CPU for plotting
x_val_plot = x_val_batch.cpu().numpy()
y_val_plot = y_val_batch.cpu().numpy()
y_pred_plot = y_pred.cpu().numpy()

plt.figure(figsize=(10, 6))
plt.scatter(x_val_plot, y_val_plot, label='Original Data', color='blue', alpha=0.5)
plt.plot(x_val_plot, y_pred_plot, label='Model Prediction', color='red', linewidth=3)
plt.title('Linear Regression with PyTorch')
plt.xlabel('Feature X work experience in (years) and max 10 years')
plt.ylabel('Target y salary in millions of rupiah(Rp)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
print("weight (w) :", model.layer.weight.item())
print("bias (b) :", model.layer.bias.item())